In [5]:
from pathlib import Path
from zipfile import ZipFile
from datetime import datetime

import polars as pl

In [6]:
# Things that the user must configure
# 1. Data Directory - the path of the folder containing the downloaded zip
# 2. NPPES file name - the exact name of the downloaded file

In [ ]:
# -------------------------------------------------------------------
# USER CONFIGURATION
# -------------------------------------------------------------------

# Directory where the downloaded NPPES file is stored.
DATA_DIR = Path(r"C:\Users\user\path\to\nppes\data")

# Name of the downloaded CMS file.
# Supported inputs:
#   1. Monthly NPPES .zip file
#   2. Already-extracted main NPPES .csv file
# Example:
# NPPES_FILE_NAME = "NPPES_Data_Dissemination_August_2026_V2.zip"

# NPPES_FILE_NAME = "NPPES_FILE.zip"
NPPES_FILE_NAME = "NPPES_Data_Dissemination_September_2026_V2.zip"

# Directory for analytical outputs created by this notebook.
OUTPUT_DIR = Path(r"C:\Users\user\path\to\nppes\outputs")

# Create output directory if it does not already exist.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_FILE = DATA_DIR / NPPES_FILE_NAME

print(f"Configured source: {SOURCE_FILE}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")

In [10]:
# -------------------------------------------------------------------
# VALIDATE INPUT FILE
# -------------------------------------------------------------------

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"DATA_DIR does not exist:\n{DATA_DIR}"
    )

if not SOURCE_FILE.exists():
    raise FileNotFoundError(
        f"NPPES source file was not found:\n{SOURCE_FILE}\n\n"
        "Download the monthly NPPES V2 file from CMS and place it "
        "in the configured DATA_DIR."
    )

supported_suffixes = {".zip", ".csv"}

if SOURCE_FILE.suffix.lower() not in supported_suffixes:
    raise ValueError(
        f"Unsupported file type: {SOURCE_FILE.suffix}\n"
        "Expected either a .zip or .csv NPPES file."
    )

file_size_gb = SOURCE_FILE.stat().st_size / (1024 ** 3)

print("Input file validation passed.")
print(f"File: {SOURCE_FILE.name}")
print(f"Size: {file_size_gb:,.2f} GB")

Input file validation passed.
File: NPPES_Data_Dissemination_September_2026_V2.zip
Size: 1.08 GB


In [ ]:
# -------------------------------------------------------------------
# RESOLVE MAIN NPPES PROVIDER CSV
# -------------------------------------------------------------------

def resolve_nppes_csv(source_file: Path, extraction_dir: Path) -> Path:
    """
    Return the path to the main NPPES provider CSV.

    If source_file is already a CSV, return it directly.

    If source_file is a ZIP:
        - identify the main npidata_pfile CSV
        - extract only that CSV
        - reuse an existing extracted copy if already present

    This avoids extracting unnecessary NPPES reference files.
    """

    source_file = source_file.resolve()

    if source_file.suffix.lower() == ".csv":
        return source_file

    extraction_dir.mkdir(parents=True, exist_ok=True)

    with ZipFile(source_file, "r") as archive:
        members = archive.namelist()

        # Main NPPES provider files conventionally contain
        # "npidata_pfile" in the filename.
        candidates = [
            member
            for member in members
            if "npidata_pfile" in Path(member).name.lower()
            and "_fileheader" not in Path(member).name.lower()
            and member.lower().endswith(".csv")
        ]

        if len(candidates) == 0:
            raise FileNotFoundError(
                "Could not identify the main NPPES provider CSV "
                "inside the ZIP archive."
            )

        if len(candidates) > 1:
            raise RuntimeError(
                "More than one main NPPES provider CSV was found:\n"
                + "\n".join(candidates)
            )

        archive_member = candidates[0]
        extracted_path = extraction_dir / Path(archive_member).name

        if extracted_path.exists():
            print("Using previously extracted provider CSV.")
            return extracted_path

        print(f"Extracting: {Path(archive_member).name}")
        archive.extract(archive_member, path=extraction_dir)

        # Handle ZIP entries that contain a nested directory.
        archive_extracted_path = extraction_dir / archive_member

        if archive_extracted_path != extracted_path:
            archive_extracted_path.replace(extracted_path)

        return extracted_path


EXTRACT_DIR = DATA_DIR / "extracted"

NPPES_CSV = resolve_nppes_csv(
    source_file=SOURCE_FILE,
    extraction_dir=EXTRACT_DIR,
)

print(f"NPPES CSV ready:\n{NPPES_CSV}")

In [14]:
# -------------------------------------------------------------------
# NPPES COLUMNS REQUIRED FOR THIS ANALYSIS
# -------------------------------------------------------------------

COL_NPI = "NPI"
COL_ENTITY_TYPE = "Entity Type Code"

COL_STATE = (
    "Provider Business Practice Location Address State Name"
)

COL_COUNTRY = (
    "Provider Business Practice Location Address Country Code (If outside U.S.)"
)

COL_DEACTIVATION_DATE = "NPI Deactivation Date"
COL_REACTIVATION_DATE = "NPI Reactivation Date"

REQUIRED_COLUMNS = [
    COL_NPI,
    COL_ENTITY_TYPE,
    COL_STATE,
    COL_COUNTRY,
    COL_DEACTIVATION_DATE,
    COL_REACTIVATION_DATE,
]

REQUIRED_COLUMNS

['NPI',
 'Entity Type Code',
 'Provider Business Practice Location Address State Name',
 'Provider Business Practice Location Address Country Code (If outside U.S.)',
 'NPI Deactivation Date',
 'NPI Reactivation Date']

In [16]:
# -------------------------------------------------------------------
# SCHEMA VALIDATION
# -------------------------------------------------------------------

schema = pl.scan_csv(
    NPPES_CSV,
    infer_schema=False
).collect_schema()

available_columns = set(schema.names())
missing_columns = set(REQUIRED_COLUMNS) - available_columns

if missing_columns:
    raise ValueError(
        "Expected NPPES columns were not found:\n"
        + "\n".join(sorted(missing_columns))
    )

print("Schema validation passed.")
print(f"Source contains {len(available_columns):,} columns.")
print(f"Analysis requires {len(REQUIRED_COLUMNS)} columns.")

Schema validation passed.
Source contains 330 columns.
Analysis requires 6 columns.


In [ ]:
# -------------------------------------------------------------------
# LAZY NPPES DATASET
# -------------------------------------------------------------------

nppes = (
    pl.scan_csv(
        NPPES_CSV,
        infer_schema=False,
        null_values=[""],
        ignore_errors=False,
    )
    .select(REQUIRED_COLUMNS)
)

nppes

In [ ]:
# -------------------------------------------------------------------
# NORMALIZE CORE ANALYTICAL FIELDS
# -------------------------------------------------------------------

nppes_clean = (
    nppes
    .with_columns(

        # NPI is an identifier, not a numeric measure.
        pl.col(COL_NPI)
        .cast(pl.String)
        .str.strip_chars()
        .alias("npi"),

        # Entity Type:
        #   1 = Individual
        #   2 = Organization
        pl.col(COL_ENTITY_TYPE)
        .cast(pl.String)
        .str.strip_chars()
        .alias("entity_type_code"),

        # Standardize state abbreviations.
        pl.col(COL_STATE)
        .cast(pl.String)
        .str.strip_chars()
        .str.to_uppercase()
        .alias("state"),

        pl.col(COL_COUNTRY)
        .cast(pl.String)
        .str.strip_chars()
        .str.to_uppercase()
        .alias("country_code"),

        # Retain status-related fields for auditability.
        pl.col(COL_DEACTIVATION_DATE)
        .cast(pl.String)
        .str.strip_chars()
        .alias("deactivation_date"),

        pl.col(COL_REACTIVATION_DATE)
        .cast(pl.String)
        .str.strip_chars()
        .alias("reactivation_date"),
    )
    .select(
        "npi",
        "entity_type_code",
        "state",
        "country_code",
        "deactivation_date",
        "reactivation_date",
    )
)

nppes_clean

In [22]:
# -------------------------------------------------------------------
# 50 STATES + DISTRICT OF COLUMBIA
# -------------------------------------------------------------------

US_STATE_CODES = [
    "AL", "AK", "AZ", "AR", "CA", "CO", "CT", "DE", "FL", "GA",
    "HI", "ID", "IL", "IN", "IA", "KS", "KY", "LA", "ME", "MD",
    "MA", "MI", "MN", "MS", "MO", "MT", "NE", "NV", "NH", "NJ",
    "NM", "NY", "NC", "ND", "OH", "OK", "OR", "PA", "RI", "SC",
    "SD", "TN", "TX", "UT", "VT", "VA", "WA", "WV", "WI", "WY",
    "DC",
]

STATE_NAME_MAP = {
    "AL": "Alabama",
    "AK": "Alaska",
    "AZ": "Arizona",
    "AR": "Arkansas",
    "CA": "California",
    "CO": "Colorado",
    "CT": "Connecticut",
    "DE": "Delaware",
    "FL": "Florida",
    "GA": "Georgia",
    "HI": "Hawaii",
    "ID": "Idaho",
    "IL": "Illinois",
    "IN": "Indiana",
    "IA": "Iowa",
    "KS": "Kansas",
    "KY": "Kentucky",
    "LA": "Louisiana",
    "ME": "Maine",
    "MD": "Maryland",
    "MA": "Massachusetts",
    "MI": "Michigan",
    "MN": "Minnesota",
    "MS": "Mississippi",
    "MO": "Missouri",
    "MT": "Montana",
    "NE": "Nebraska",
    "NV": "Nevada",
    "NH": "New Hampshire",
    "NJ": "New Jersey",
    "NM": "New Mexico",
    "NY": "New York",
    "NC": "North Carolina",
    "ND": "North Dakota",
    "OH": "Ohio",
    "OK": "Oklahoma",
    "OR": "Oregon",
    "PA": "Pennsylvania",
    "RI": "Rhode Island",
    "SC": "South Carolina",
    "SD": "South Dakota",
    "TN": "Tennessee",
    "TX": "Texas",
    "UT": "Utah",
    "VT": "Vermont",
    "VA": "Virginia",
    "WA": "Washington",
    "WV": "West Virginia",
    "WI": "Wisconsin",
    "WY": "Wyoming",
    "DC": "District of Columbia",
}

assert len(US_STATE_CODES) == 51
assert len(STATE_NAME_MAP) == 51

In [24]:
# -------------------------------------------------------------------
# SOURCE DATA QUALITY SUMMARY
# -------------------------------------------------------------------

source_quality = (
    nppes_clean
    .select(
        pl.len().alias("source_rows"),

        pl.col("npi")
        .n_unique()
        .alias("unique_npis"),

        pl.col("entity_type_code")
        .eq("1")
        .sum()
        .alias("type_1_rows"),

        pl.col("entity_type_code")
        .eq("2")
        .sum()
        .alias("type_2_rows"),

        (pl.col("entity_type_code").is_null()
            | ~pl.col("entity_type_code").is_in(["1", "2"]).fill_null(False)
        )
        .sum()
        .alias("rows_without_valid_entity_type"),
        
        pl.col("state")
        .is_null()
        .sum()
        .alias("rows_missing_state"),

        pl.col("deactivation_date")
        .is_not_null()
        .sum()
        .alias("rows_with_deactivation_date"),
    )
    .collect()
)

source_quality

source_rows,unique_npis,type_1_rows,type_2_rows,rows_without_valid_entity_type,rows_missing_state,rows_with_deactivation_date
u32,u32,u32,u32,u32,u32,u32
9798758,9798758,7471371,1972058,355329,355331,374443


In [25]:
# -------------------------------------------------------------------
# NPI UNIQUENESS CHECK
# -------------------------------------------------------------------

quality = source_quality.row(0, named=True)

if quality["source_rows"] != quality["unique_npis"]:
    print(
        "WARNING: Source row count does not equal unique NPI count."
    )
    print(
        f"Rows:        {quality['source_rows']:,}\n"
        f"Unique NPIs: {quality['unique_npis']:,}"
    )
else:
    print(
        f"NPI uniqueness check passed: "
        f"{quality['unique_npis']:,} unique NPIs."
    )

NPI uniqueness check passed: 9,798,758 unique NPIs.


In [27]:
# -------------------------------------------------------------------
# VALID ENTITY POPULATION
# -------------------------------------------------------------------

# A usable Type 1 / Type 2 analytical record must have one of the
# two defined NPPES Entity Type Codes.
#
# CMS states:
#   1 = Individual
#   2 = Organization

valid_entities = nppes_clean.filter(
    pl.col("entity_type_code").is_in(["1", "2"])
)

valid_entities

In [28]:
# -------------------------------------------------------------------
# NATIONAL ENTITY-TYPE COUNTS
# -------------------------------------------------------------------

national_summary = (
    valid_entities
    .select(
        pl.col("entity_type_code")
        .eq("1")
        .sum()
        .alias("type_1_npis"),

        pl.col("entity_type_code")
        .eq("2")
        .sum()
        .alias("type_2_npis"),

        pl.len()
        .alias("total_npis"),
    )
    .with_columns(
        (
            pl.col("type_1_npis")
            / pl.col("total_npis")
            * 100
        )
        .alias("type_1_pct"),

        (
            pl.col("type_2_npis")
            / pl.col("total_npis")
            * 100
        )
        .alias("type_2_pct"),
    )
    .collect()
)

national_summary

type_1_npis,type_2_npis,total_npis,type_1_pct,type_2_pct
u32,u32,u32,f64,f64
7471371,1972058,9443429,79.117141,20.882859


In [29]:
# -------------------------------------------------------------------
# REPORT NATIONAL RESULTS
# -------------------------------------------------------------------

national = national_summary.row(0, named=True)

print("NPPES NATIONAL SUMMARY")
print("-" * 45)

print(
    f"Type 1 — Individuals:    "
    f"{national['type_1_npis']:>12,} "
    f"({national['type_1_pct']:.1f}%)"
)

print(
    f"Type 2 — Organizations:  "
    f"{national['type_2_npis']:>12,} "
    f"({national['type_2_pct']:.1f}%)"
)

print("-" * 45)

print(
    f"Total Type 1 + Type 2:   "
    f"{national['total_npis']:>12,}"
)

NPPES NATIONAL SUMMARY
---------------------------------------------
Type 1 — Individuals:       7,471,371 (79.1%)
Type 2 — Organizations:     1,972,058 (20.9%)
---------------------------------------------
Total Type 1 + Type 2:      9,443,429


In [31]:
# -------------------------------------------------------------------
# 50 STATES + DC ANALYTICAL POPULATION
# -------------------------------------------------------------------

us_state_entities = valid_entities.filter(
    pl.col("state").is_in(US_STATE_CODES)
)
us_state_entities

In [32]:
# -------------------------------------------------------------------
# U.S. STATE ALLOCATION DENOMINATOR
# -------------------------------------------------------------------

us_state_total = (
    us_state_entities
    .select(
        pl.len().alias("us_state_npis")
    )
    .collect()
    .item()
)

national_total = national["total_npis"]

outside_state_allocation = national_total - us_state_total

print(f"National Type 1 + Type 2 NPIs: {national_total:,}")
print(f"50 states + DC NPIs:            {us_state_total:,}")
print(f"Outside 50 states + DC:         {outside_state_allocation:,}")

National Type 1 + Type 2 NPIs: 9,443,429
50 states + DC NPIs:            9,371,048
Outside 50 states + DC:         72,381


In [48]:
# -------------------------------------------------------------------
# STATE-LEVEL NPI AGGREGATION
# -------------------------------------------------------------------

state_summary = (
    us_state_entities

    .group_by("state")

    .agg(
        pl.col("entity_type_code")
        .eq("1")
        .sum()
        .alias("type_1_npis"),

        pl.col("entity_type_code")
        .eq("2")
        .sum()
        .alias("type_2_npis"),

        pl.len()
        .alias("total_npis"),
    )

    .with_columns(
        # Percentage of the 50-state + DC NPI population.
        (
            pl.col("total_npis")
            / pl.lit(us_state_total)
            * 100
        )
        .alias("allocation_pct"),

        # Entity-type composition inside each state.
        (
            pl.col("type_1_npis")
            / pl.col("total_npis")
            * 100
        )
        .alias("type_1_pct"),

        (
            pl.col("type_2_npis")
            / pl.col("total_npis")
            * 100
        )
        .alias("type_2_pct"),
    )

    .sort(
        "total_npis",
        descending=True,
    )

    .with_row_index(
        name="rank",
        offset=1,
    )

    .collect()
)

# state_summary = state_summary.with_columns(
#     pl.col("allocation_pct").round(4),
#     pl.col("type_1_pct").round(4),
#     pl.col("type_2_pct").round(4),
# )

state_summary

rank,state,type_1_npis,type_2_npis,total_npis,allocation_pct,type_1_pct,type_2_pct
u32,str,u32,u32,u32,f64,f64,f64
1,"""CA""",990637,197549,1188186,12.679329,83.373899,16.626101
2,"""NY""",553773,104805,658578,7.027795,84.086167,15.913833
3,"""FL""",480260,174057,654317,6.982325,73.398674,26.601326
4,"""TX""",441789,167009,608798,6.496584,72.56742,27.43258
5,"""OH""",341771,66293,408064,4.354518,83.754264,16.245736
…,…,…,…,…,…,…,…
47,"""DE""",21634,6341,27975,0.298526,77.333333,22.666667
48,"""ND""",22851,4818,27669,0.29526,82.587011,17.412989
49,"""SD""",17646,5571,23217,0.247752,76.004652,23.995348


In [50]:
# -------------------------------------------------------------------
# ADD HUMAN-READABLE STATE NAMES
# -------------------------------------------------------------------

state_lookup = pl.DataFrame(
    {
        "state": list(STATE_NAME_MAP.keys()),
        "state_name": list(STATE_NAME_MAP.values()),
    }
)

state_summary = (
    state_summary
    .join(
        state_lookup,
        on="state",
        how="left",
    )
    .select(
        "rank",
        "state",
        "state_name",
        "type_1_npis",
        "type_2_npis",
        "total_npis",
        "allocation_pct",
        "type_1_pct",
        "type_2_pct",
    )
)

state_summary

rank,state,state_name,type_1_npis,type_2_npis,total_npis,allocation_pct,type_1_pct,type_2_pct
u32,str,str,u32,u32,u32,f64,f64,f64
1,"""CA""","""California""",990637,197549,1188186,12.679329,83.373899,16.626101
2,"""NY""","""New York""",553773,104805,658578,7.027795,84.086167,15.913833
3,"""FL""","""Florida""",480260,174057,654317,6.982325,73.398674,26.601326
4,"""TX""","""Texas""",441789,167009,608798,6.496584,72.56742,27.43258
5,"""OH""","""Ohio""",341771,66293,408064,4.354518,83.754264,16.245736
…,…,…,…,…,…,…,…,…
47,"""DE""","""Delaware""",21634,6341,27975,0.298526,77.333333,22.666667
48,"""ND""","""North Dakota""",22851,4818,27669,0.29526,82.587011,17.412989
49,"""SD""","""South Dakota""",17646,5571,23217,0.247752,76.004652,23.995348


In [51]:
# -------------------------------------------------------------------
# STATE COVERAGE VALIDATION
# -------------------------------------------------------------------

observed_states = set(
    state_summary["state"].to_list()
)

expected_states = set(US_STATE_CODES)

missing_states = expected_states - observed_states
unexpected_states = observed_states - expected_states

if missing_states:
    print(
        "WARNING — expected states missing from output:",
        sorted(missing_states),
    )
else:
    print("All 50 states + DC are represented.")

if unexpected_states:
    print(
        "WARNING — unexpected geography detected:",
        sorted(unexpected_states),
    )

All 50 states + DC are represented.


In [52]:
# -------------------------------------------------------------------
# RECONCILE STATE-LEVEL TOTALS
# -------------------------------------------------------------------

state_type_1_total = state_summary["type_1_npis"].sum()
state_type_2_total = state_summary["type_2_npis"].sum()
state_grand_total = state_summary["total_npis"].sum()

assert state_grand_total == us_state_total

assert (
    state_type_1_total
    + state_type_2_total
    == state_grand_total
)

print("State-level reconciliation passed.")
print()
print(f"Type 1 NPIs: {state_type_1_total:,}")
print(f"Type 2 NPIs: {state_type_2_total:,}")
print(f"Total NPIs:  {state_grand_total:,}")

State-level reconciliation passed.

Type 1 NPIs: 7,418,484
Type 2 NPIs: 1,952,564
Total NPIs:  9,371,048


In [53]:
# -------------------------------------------------------------------
# PERCENTAGE VALIDATION
# -------------------------------------------------------------------

allocation_total = state_summary["allocation_pct"].sum()

print(
    f"State allocation percentages sum to: "
    f"{allocation_total:.8f}%"
)

assert abs(allocation_total - 100.0) < 0.000001

entity_mix_check = (
    state_summary
    .with_columns(
        (
            pl.col("type_1_pct")
            + pl.col("type_2_pct")
        )
        .alias("entity_mix_total")
    )
)

max_mix_error = (
    entity_mix_check["entity_mix_total"] - 100
).abs().max()

assert max_mix_error < 0.000001

print("Percentage reconciliation passed.")

State allocation percentages sum to: 100.00000000%
Percentage reconciliation passed.


In [54]:
# -------------------------------------------------------------------
# PRESENTATION TABLE
# -------------------------------------------------------------------

state_report = (
    state_summary
    .with_columns(
        pl.col("allocation_pct").round(2),
        pl.col("type_1_pct").round(2),
        pl.col("type_2_pct").round(2),
    )
)

state_report

rank,state,state_name,type_1_npis,type_2_npis,total_npis,allocation_pct,type_1_pct,type_2_pct
u32,str,str,u32,u32,u32,f64,f64,f64
1,"""CA""","""California""",990637,197549,1188186,12.68,83.37,16.63
2,"""NY""","""New York""",553773,104805,658578,7.03,84.09,15.91
3,"""FL""","""Florida""",480260,174057,654317,6.98,73.4,26.6
4,"""TX""","""Texas""",441789,167009,608798,6.5,72.57,27.43
5,"""OH""","""Ohio""",341771,66293,408064,4.35,83.75,16.25
…,…,…,…,…,…,…,…,…
47,"""DE""","""Delaware""",21634,6341,27975,0.3,77.33,22.67
48,"""ND""","""North Dakota""",22851,4818,27669,0.3,82.59,17.41
49,"""SD""","""South Dakota""",17646,5571,23217,0.25,76.0,24.0


In [55]:
# -------------------------------------------------------------------
# TOP 10 STATES BY NPI POPULATION
# -------------------------------------------------------------------

state_report.head(10)

rank,state,state_name,type_1_npis,type_2_npis,total_npis,allocation_pct,type_1_pct,type_2_pct
u32,str,str,u32,u32,u32,f64,f64,f64
1,"""CA""","""California""",990637,197549,1188186,12.68,83.37,16.63
2,"""NY""","""New York""",553773,104805,658578,7.03,84.09,15.91
3,"""FL""","""Florida""",480260,174057,654317,6.98,73.4,26.6
4,"""TX""","""Texas""",441789,167009,608798,6.5,72.57,27.43
5,"""OH""","""Ohio""",341771,66293,408064,4.35,83.75,16.25
6,"""MI""","""Michigan""",292168,65873,358041,3.82,81.6,18.4
7,"""PA""","""Pennsylvania""",265654,70794,336448,3.59,78.96,21.04
8,"""IL""","""Illinois""",244871,74196,319067,3.4,76.75,23.25
9,"""NC""","""North Carolina""",204222,73423,277645,2.96,73.56,26.44


In [ ]:
# -------------------------------------------------------------------
# EXPORT NATIONAL SUMMARY
# -------------------------------------------------------------------

national_output = OUTPUT_DIR / "nppes_national_summary.csv"

national_summary.write_csv(national_output)

print(f"Saved: {national_output.resolve()}")

In [ ]:
# -------------------------------------------------------------------
# EXPORT STATE MARKET SUMMARY
# -------------------------------------------------------------------

state_output = OUTPUT_DIR / "nppes_state_summary.csv"

state_report.write_csv(state_output)

print(f"Saved: {state_output.resolve()}")

In [ ]:
# -------------------------------------------------------------------
# OPTIONAL PARQUET EXPORT
# -------------------------------------------------------------------

parquet_output = OUTPUT_DIR / "nppes_state_summary.parquet"

state_summary.write_parquet(parquet_output)

print(f"Saved: {parquet_output.resolve()}")

In [59]:
# -------------------------------------------------------------------
# ANALYSIS METADATA
# -------------------------------------------------------------------

run_metadata = pl.DataFrame(
    {
        "analysis_timestamp": [
            datetime.now().isoformat(timespec="seconds")
        ],
        "source_file": [
            SOURCE_FILE.name
        ],
        "resolved_csv": [
            NPPES_CSV.name
        ],
        "national_type_1_npis": [
            national["type_1_npis"]
        ],
        "national_type_2_npis": [
            national["type_2_npis"]
        ],
        "national_total_npis": [
            national["total_npis"]
        ],
        "fifty_states_plus_dc_npis": [
            us_state_total
        ],
    }
)

metadata_output = OUTPUT_DIR / "analysis_metadata.csv"

run_metadata.write_csv(metadata_output)

run_metadata

analysis_timestamp,source_file,resolved_csv,national_type_1_npis,national_type_2_npis,national_total_npis,fifty_states_plus_dc_npis
str,str,str,i64,i64,i64,i64
"""2026-09-14T11:35:51""","""NPPES_Data_Dissemination_Septe…","""npidata_pfile_20050523-2026091…",7471371,1972058,9443429,9371048


In [60]:
# -------------------------------------------------------------------
# FINAL ANALYSIS SUMMARY
# -------------------------------------------------------------------

top_state = state_summary.row(0, named=True)

print("=" * 60)
print("NPPES PROVIDER MARKET ANALYSIS COMPLETE")
print("=" * 60)

print()
print("NATIONAL NPI POPULATION")
print(
    f"Type 1 Individuals:   "
    f"{national['type_1_npis']:,}"
)
print(
    f"Type 2 Organizations: "
    f"{national['type_2_npis']:,}"
)
print(
    f"Combined:             "
    f"{national['total_npis']:,}"
)

print()
print("STATE MARKET ANALYSIS")
print(
    f"50 states + DC:       "
    f"{us_state_total:,}"
)

print(
    f"Largest NPI market:   "
    f"{top_state['state_name']}"
)

print(
    f"Market NPIs:          "
    f"{top_state['total_npis']:,}"
)

print(
    f"U.S. allocation:      "
    f"{top_state['allocation_pct']:.2f}%"
)

print()
print("OUTPUT FILES")
print(f"- {national_output}")
print(f"- {state_output}")
print(f"- {parquet_output}")
print(f"- {metadata_output}")

NPPES PROVIDER MARKET ANALYSIS COMPLETE

NATIONAL NPI POPULATION
Type 1 Individuals:   7,471,371
Type 2 Organizations: 1,972,058
Combined:             9,443,429

STATE MARKET ANALYSIS
50 states + DC:       9,371,048
Largest NPI market:   California
Market NPIs:          1,188,186
U.S. allocation:      12.68%

OUTPUT FILES
- outputs\nppes_national_summary.csv
- outputs\nppes_state_summary.csv
- outputs\nppes_state_summary.parquet
- outputs\analysis_metadata.csv
